# 02 - Cleaning and preprocessing

Turns the collected question files (`data/raw/questions_<tag>.parquet`) plus the enrichment tables (`answers.parquet`, `users.parquet`) into one analysis-ready table. Every step is a function in `src/cleaning.py`; this notebook runs them in order, shows what changed, and gives the reason for each decision.

Licence note: Stack Overflow content is CC BY-SA 4.0. Examples below are short snippets for illustration only; raw text is not published.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
from src import cleaning

pd.set_option("display.max_columns", 60, "display.width", 150, "display.max_colwidth", 120)
COLLECTION_DATE = "2026-10-03"   # UTC date the question collection finished (Member 1, M1-07)
files = sorted((ROOT / "data" / "raw").glob("questions_*.parquet"))
frames = [pd.read_parquet(f) for f in files]
answers = pd.read_parquet(ROOT / "data" / "processed" / "answers.parquet")
users = pd.read_parquet(ROOT / "data" / "processed" / "users.parquet")
pd.DataFrame({"file": [f.name for f in files], "rows": [len(f_) for f_ in frames]})

,file,rows
0,questions_python.parquet,84574


## M2-05 Merge and deduplication

A question carrying two selected tags appears in two tag files. We keep one row per `question_id`, keep the full list of tags it was collected under, and keep the **smaller** `sampling_weight` when the copies differ (the smaller weight is the more conservative estimate of how many real questions the row stands for). Owner IDs are hashed with the project salt and joined to `users.parquet`; the raw `owner_user_id` is dropped so it never reaches the processed table.

In [2]:
q, s_merge = cleaning.merge_questions(frames, answers, users)
pd.Series(s_merge, name="count").to_frame()

,count
rows_before,84574
weight_conflicts,0
rows_after,84574
duplicates_removed,0


## M2-06 Timestamp conversion and checks

All Unix times are converted to UTC datetimes. Checks: no answer earlier than its question, no question outside January 2019 to August 2026, and no account created after the question it asked.

In [3]:
q, s_time = cleaning.convert_timestamps(q)
display(pd.Series(s_time, name="rows with problem").to_frame())
bad = q[(q.first_answer_at < q.created_at) | (q.account_created_at > q.created_at)]
bad.assign(first_answer_minus_question_s=(bad.first_answer_at - bad.created_at).dt.total_seconds(),
           account_minus_question_s=(bad.account_created_at - bad.created_at).dt.total_seconds()
          )[["question_id", "created_at", "first_answer_minus_question_s", "account_minus_question_s"]]

,rows with problem
accepted_before_question,0
first_answer_before_question,1
accepted_before_first_answer,0
question_outside_window,0
account_after_question,4


,question_id,created_at,first_answer_minus_question_s,account_minus_question_s
16665,60814604,2020-03-22 21:52:47+00:00,-3074.0,-25593698.0
28391,64408183,2020-10-17 23:05:05+00:00,1395501.0,106358086.0
61253,75496612,2023-02-17 14:39:17+00:00,380644.0,245552.0
65511,77325097,2023-10-19 15:20:13+00:00,NaN,11332698.0
81274,79804579,2025-10-29 17:48:46+00:00,NaN,56775.0


**Decision.** The few exceptions are kept; none is removed on the basis of a target. The cause is not known from the API data alone (moderator-merged or migrated accounts are plausible, but not verified). How each is handled: a negative time to first answer still counts as answered within 7 days, so the label is unaffected; an asker account newer than its question would give a negative account age, so account age is clipped at 0 days (see M2-08).

## M2-07 HTML cleaning and code extraction

Question bodies are HTML. With BeautifulSoup, every `<pre>` block and every inline `<code>` is moved into `code_text`; `plain_body` is the remaining prose. Derived fields: `n_code_blocks` (number of `<pre>` blocks), `code_ratio` (code characters over code plus prose characters) and `has_error_text` (contains Traceback, Exception, `Error:` or errno).

In [4]:
q = cleaning.add_text_features(q)
ex = q.sample(3, random_state=42)
for _, r in ex.iterrows():
    print(f"question_id {r.question_id} | n_code_blocks {r.n_code_blocks} | code_ratio {r.code_ratio:.2f} | has_error_text {r.has_error_text}")
    print("  BEFORE (html)  :", r.body[:180].replace("\n", " "))
    print("  AFTER plain    :", r.plain_body[:180])
    print("  AFTER code     :", r.code_text[:120].replace("\n", " "))
    print()
q[["n_code_blocks", "code_ratio", "has_error_text"]].describe().T[["mean", "50%", "max"]].round(3)

question_id 74338540 | n_code_blocks 0 | code_ratio 0.00 | has_error_text False
  BEFORE (html)  : <p>I'm new to python and I want to calculate the sum of the daily average temperatures on many hours and then append it to the dataframe  e.g :</p> <div class="s-table-container"> 
  AFTER plain    : I'm new to python and I want to calculate the sum of the daily average temperatures on many hours and then append it to the dataframe e.g : date Temperature 2015-04-01 00:00:00 3.9
  AFTER code     : 

question_id 55212737 | n_code_blocks 1 | code_ratio 0.54 | has_error_text False
  BEFORE (html)  : <p>I want to scrape the data of websitses using Beautiful Soup and requests, and I've come so far that I've got the data I want but now I want to filter it:</p>  <pre><code>from bs
  AFTER plain    : I want to scrape the data of websitses using Beautiful Soup and requests, and I've come so far that I've got the data I want but now I want to filter it: Let's say this prints the 
  AFTER code     : 

,mean,50%,max
n_code_blocks,1.828,2.00,39.0
code_ratio,0.521,0.56,1.0


## M2-08 Missing values

- **No accepted answer** is not missing data: it means the question was not resolved, so `target_resolved_7d` is 0.
- **No answer** means `target_answered_7d` is 0 for the same reason.
- **Deleted or unknown asker**: the account creation date does not exist, so account age is imputed with the median account age of questions posted in the training period (before January 2025), and `missing_account_age` records which rows were imputed. Using only the training period avoids leaking later information.

In [5]:
before = {
    "no accepted answer": int(q.accepted_answer_creation_date.isna().sum()),
    "no answer at all": int(q.first_answer_creation_date.isna().sum()),
    "no account creation date": int(q.account_creation_date.isna().sum()),
}
q, s_age = cleaning.add_account_age(q)
pd.DataFrame({
    "missing before": before,
    "missing after": {"no accepted answer": "kept: means target 0", "no answer at all": "kept: means target 0",
                       "no account creation date": int(q.account_age_days.isna().sum())},
    "treatment": {"no accepted answer": "not missing", "no answer at all": "not missing",
                  "no account creation date": f"median {s_age['account_age_median_days']:.0f} days + flag"},
})

,missing before,missing after,treatment
no accepted answer,47339,kept: means target 0,not missing
no answer at all,15205,kept: means target 0,not missing
no account creation date,1837,0,median 374 days + flag


## M2-09 Both targets and label maturity

- Main target: `hours_to_accept` and `target_resolved_7d` (accepted answer within 168 hours).
- Secondary target: `hours_to_first_answer` and `target_answered_7d` (any answer within 168 hours).
- `accepted_given_answered`: among questions answered within 7 days, whether one was also accepted within 7 days. A fall over time would mean people accept less, which would change how the main target should be read.
- **Maturity rule:** questions posted fewer than 30 days before the collection date are dropped so every label has had time to form.

In [6]:
q, s_target = cleaning.add_targets(q, COLLECTION_DATE)
print(s_target)
def balance(col):
    c = q[col].value_counts().sort_index()
    w = q.groupby(col).sampling_weight.sum()
    return pd.DataFrame({"rows": c, "share": (c / c.sum()).round(3), "weighted share": (w / w.sum()).round(3)})
display(balance("target_resolved_7d"))
display(balance("target_answered_7d"))
print("accepted_given_answered mean:", round(q.accepted_given_answered.mean(), 3))

{'immature_dropped': 0, 'maturity_cutoff': '2026-09-03'}


,rows,share,weighted share
target_resolved_7d,,,
0,49140,0.581,0.563
1,35434,0.419,0.437


,rows,share,weighted share
target_answered_7d,,,
0,20347,0.241,0.233
1,64227,0.759,0.767


accepted_given_answered mean: 0.552


## M2-10 Invalid and extreme records

| Check | Decision | Reason |
|---|---|---|
| Empty body (no prose and no code) | remove | nothing to model |
| More than half of the letters are non-ASCII | remove | text features are built for English; the sample is tiny |
| Spam-like: under 30 characters of text and a link | remove | no question content |
| Very long body (above the 99.9th percentile of prose length) | cap the prose at that length, keep the row | the question itself is valid; extreme lengths would dominate TF-IDF |

No decision uses either target.

In [7]:
q, s_invalid = cleaning.flag_invalid(q)
pd.Series(s_invalid, name="value").to_frame()

,value
empty_removed,0
non_english_removed,2
spam_like_removed,0
long_body_cap_chars,3767
long_body_capped,85
invalid_removed,2


## M2-11 Before and after summary

In [8]:
summary = pd.DataFrame([
    ("Rows collected (all tag files)", s_merge["rows_before"]),
    ("Duplicates removed (multi-tag questions)", s_merge["duplicates_removed"]),
    ("Rows after merge", s_merge["rows_after"]),
    ("Missing account age (imputed, flagged)", s_age["missing_account_age"]),
    ("Immature labels dropped (< 30 days old)", s_target["immature_dropped"]),
    ("Invalid records removed", s_invalid["invalid_removed"]),
    ("Long bodies capped (kept)", s_invalid["long_body_capped"]),
    ("Final rows", len(q)),
    ("target_resolved_7d = 1 (share)", round(q.target_resolved_7d.mean(), 3)),
    ("target_answered_7d = 1 (share)", round(q.target_answered_7d.mean(), 3)),
], columns=["step", "value"])
summary["value"] = summary["value"].map(lambda v: f"{int(v):,}" if float(v).is_integer() else f"{v:.3f}")
summary

,step,value
0,Rows collected (all tag files),"84,574"
1,Duplicates removed (multi-tag questions),0
2,Rows after merge,"84,574"
3,"Missing account age (imputed, flagged)","1,837"
4,Immature labels dropped (< 30 days old),0
5,Invalid records removed,2
6,Long bodies capped (kept),85
7,Final rows,"84,572"
8,target_resolved_7d = 1 (share),0.419
9,target_answered_7d = 1 (share),0.759


## M2-12 Save processed dataset v1

`questions_clean_v1.parquet` is only written when every configured tag is present, so a partial run never gets mistaken for the full dataset. The raw HTML `body` is dropped from the saved table (prose is in `plain_body`, code in `code_text`) to keep it small.

In [9]:
import yaml
expected = set(yaml.safe_load(open(ROOT / "config.yaml"))["study"]["tags"])
present = set(t for ts in q.collected_tags for t in ts)
missing_tags = sorted(expected - present)
if missing_tags:
    print("NOT saved: tag files missing for", missing_tags)
else:
    out = ROOT / "data" / "processed" / "questions_clean_v1.parquet"
    q.drop(columns=["body"]).to_parquet(out, index=False)
    print("saved", out.name, len(q), "rows")

NOT saved: tag files missing for ['android', 'docker', 'excel', 'fastapi', 'java', 'javascript', 'openai-api', 'pandas', 'powerbi', 'r', 'reactjs', 'sql']
